# Benchmark Harness — Interactive Analysis

Explore benchmark results without manually reading JSONL. Compare current runs to previous runs, identify regressions visually.

## Setup

Install analysis dependencies first:

```bash
pip install bench-harness[analysis]
```

## Quick Start

Point this notebook at any `benchmark.db` produced by the harness.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from bench_harness.analysis import (
    runs_to_df,
    timing_summary_df,
    quantization_comparison_df,
    context_degradation_df,
    score_variance_df,
    style_comparison_df,
    failures_df,
    identify_mismatches,
)

## 1. Load Data

In [ ]:
# Change this to your benchmark.db path
DB_PATH = "runs/2026-05-06-coding_benchmark/benchmark.db"
df = runs_to_df(DB_PATH)
print(f"Loaded {len(df)} runs, {df['model_alias'].nunique()} models")
print(f"Columns: {list(df.columns)}")

## 2. Score vs Latency Scatter

Each dot is a task run. Larger dots = more tokens. Shows the quality/speed tradeoff.

In [ ]:
plt.figure(figsize=(10, 6))
for model, group in df.groupby('model_alias'):
    scored = group[group['score_primary'].notna()]
    plt.scatter(
        scored['tokens_per_second'],
        scored['score_primary'],
        s=scored['completion_tokens'] / 5,
        alpha=0.7,
        label=model,
    )
plt.xlabel('Tokens/sec (generation)')
plt.ylabel('Primary Score')
plt.title('Score vs Latency by Model')
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Per-Model Summary Table

In [ ]:
timing = timing_summary_df(df)
pd.set_option('display.float_format', '{:.2f}'.format)
print(timing.to_string(index=False))

## 4. Quantization Comparison

Do different quantization levels hurt quality?

In [ ]:
quant = quantization_comparison_df(df)
if not quant.empty:
    pivot = quant.pivot_table(
        index='model_alias',
        columns='quantization',
        values='avg_score',
        aggfunc='mean',
    )
    print(pivot.to_string())
    
    # Bar chart: score by quantization
    plt.figure(figsize=(10, 5))
    sns.barplot(data=quant, x='model_alias', y='avg_score', hue='quantization', capsize=0.1)
    plt.xlabel('Model')
    plt.ylabel('Average Score')
    plt.title('Score by Quantization Level')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('No quantization data in this run.')

## 5. Context Degradation Curves

Does quality drop as context length increases?

In [ ]:
ctx = context_degradation_df(df)
if not ctx.empty:
    plt.figure(figsize=(10, 6))
    for model, group in ctx.groupby('model_alias'):
        plt.errorbar(
            group['context_bucket'],
            group['avg_score'],
            yerr=group['score_std'],
            capsize=4,
            label=model,
            marker='o',
        )
    plt.xlabel('Context Size Bucket')
    plt.ylabel('Average Score')
    plt.title('Quality vs Context Length')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(ctx.to_string(index=False))
else:
    print('No context data in this run.')

## 6. Discriminating Tasks

Which tasks best differentiate models? High variance = the task is sensitive to model capability.

In [ ]:
var = score_variance_df(df)
if not var.empty:
    top = var.head(20)
    pd.set_option('display.max_columns', None)
    print(top[['task_id', 'variance', 'score_range', 'best_model', 'best_score',
                'worst_model', 'worst_score']].to_string(index=False))
else:
    print('No variance data.')

## 7. Prompt Style Comparison

If runs include different prompt styles (plain, repl, terse, etc.), compare them.

In [ ]:
styles = style_comparison_df(df)
if not styles.empty:
    # Overall score by style
    style_summary = styles.groupby('prompt_style').agg(
        avg_score=('avg_score', 'mean'),
        run_count=('run_count', 'sum'),
        avg_wall=('avg_wall_ms', 'mean'),
        avg_tok=('avg_tokens', 'mean'),
    ).sort_values('avg_score', ascending=False)
    print(style_summary.to_string())
    
    # Heatmap: task x style score
    pivot = styles.pivot_table(
        index='task_id',
        columns='prompt_style',
        values='avg_score',
    )
    if len(pivot) > 0 and len(pivot.columns) > 0:
        plt.figure(figsize=(max(10, len(pivot.columns) * 1.5), max(6, len(pivot) * 0.3)))
        sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0.5)
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.title('Score by Task and Prompt Style')
        plt.tight_layout()
        plt.show()
else:
    print('No prompt style data. Run with --styles to generate.')

## 8. Failure Clustering

Group failed runs by error pattern to spot systematic issues.

In [ ]:
fails = failures_df(df)
if not fails.empty:
    cluster_counts = fails['error_cluster'].value_counts()
    print("Error clusters:")
    for cluster, count in cluster_counts.items():
        print(f"  {cluster[:60]}... ({count} runs)")
    print()
    print(fails[['error_cluster', 'model_alias', 'task_id']].to_string(index=False))
else:
    print('No failures in this run.')

## 9. Model Identity Verification

Check that model aliases match the actual model served by the backend.
This catches cases like `qwen-dense` being served as `agent-code`.

In [ ]:
mismatches = identify_mismatches(df)
if not mismatches.empty:
    print(mismatches.to_string(index=False))
    flagged = mismatches[mismatches['mismatch_detected']]
    if not flagged.empty:
        print(f"\n{'!'*40}")
        print(f"⚠️  {len(flagged)} alias mismatch(es) detected!")
        print(f"{'!'*40}")
    else:
        print("\nAll aliases match actual served models.")
else:
    print('No identity stamp data. Run with latest harness version.')

## 10. Score Distribution by Model

Box plots of primary scores per model show spread and outliers.

In [ ]:
scored = df[df['score_primary'].notna()]
if not scored.empty:
    plt.figure(figsize=(8, 5))
    data = [scored[scored['model_alias'] == m]['score_primary'].values 
            for m in scored['model_alias'].unique()]
    bp = plt.boxplot(data, labels=scored['model_alias'].unique(), vert=True)
    plt.ylabel('Primary Score')
    plt.title('Score Distribution by Model')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('No scored runs in this dataset.')

## 11. Quick DuckDB Analysis

For large datasets, use DuckDB for faster SQL analysis.

In [ ]:
from bench_harness.analysis import query_duck

# Example: top 10 fastest models per task
result = query_duck("""
    SELECT task_id, model_alias, 
           ROUND(AVG(tokens_per_second), 1) as avg_tps,
           ROUND(AVG(total_wall_ms), 0) as avg_wall_ms,
           COUNT(*) as runs
    FROM runs 
    WHERE tokens_per_second > 0
    GROUP BY task_id, model_alias
    ORDER BY avg_tps DESC
    LIMIT 10
""", DB_PATH)
print(result.to_string(index=False))